In [ ]:
%sql
-- silver generation: 15/60 min -> hourly (the average MW over one hour is the energy in MWh)
USE CATALOG energy;

CREATE OR REPLACE TABLE d1_silver.silver_generation_hourly
USING DELTA
AS
SELECT
  g.zone_code,
  date_trunc('HOUR', g.ts_utc) AS ts_utc,
  COALESCE(p.psr_code, 'B20') AS psr_code,   -- unknown production type -> 'Other'
  AVG(g.generation_mw) AS generation_mwh,
  COUNT(*) AS n_points
FROM d1_bronze.bronze_generation g
LEFT JOIN d1_gold.dim_production_type p
  ON g.production_type = p.production_type
GROUP BY g.zone_code, date_trunc('HOUR', g.ts_utc), COALESCE(p.psr_code, 'B20');

In [ ]:
%sql
-- silver load
CREATE OR REPLACE TABLE d1_silver.silver_load_hourly
USING DELTA
AS
SELECT
  zone_code,
  date_trunc('HOUR', ts_utc) AS ts_utc,
  AVG(load_mw) AS load_mwh,
  COUNT(*) AS n_points
FROM d1_bronze.bronze_load
GROUP BY zone_code, date_trunc('HOUR', ts_utc);

In [ ]:
%sql
-- silver flows
CREATE OR REPLACE TABLE d1_silver.silver_flows_hourly
USING DELTA
AS
SELECT
  from_zone,
  to_zone,
  date_trunc('HOUR', ts_utc) AS ts_utc,
  AVG(flow_mw) AS flow_mwh
FROM d1_bronze.bronze_flows
GROUP BY from_zone, to_zone, date_trunc('HOUR', ts_utc);

In [ ]:
%sql
-- production types not found in dim_production_type (counted as 'Other', should be empty)
SELECT DISTINCT production_type
FROM d1_bronze.bronze_generation
WHERE production_type NOT IN (SELECT production_type FROM d1_gold.dim_production_type);

In [ ]:
%sql
-- incomplete hours per zone (a 15-min zone should have 4 points per hour)
SELECT zone_code, n_points, COUNT(*) AS n_hours
FROM d1_silver.silver_load_hourly
GROUP BY zone_code, n_points
ORDER BY zone_code, n_points;